In [0]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp
from pyspark.sql.types import IntegerType

# 1. Define Paths and Table Names
# We point to the 'silver' container for physical storage
silver_location = "abfss://silver@stlinggarprojectdev001.dfs.core.windows.net/nyc_ridehailing_silver"
silver_table_name = "dev_silver.default.silver_ridehailing"

# 2. Read from the Bronze Table you just created
df_bronze = spark.read.table("dev_bronze.default.bronze_ridehailing")

# 3. Apply transformations (Cleaning & Casting)
df_silver = (df_bronze
    # Remove rows with missing essential IDs
    .filter(col("hvfhs_license_num").isNotNull())
    
    # Convert strings to actual Timestamps
    .withColumn("pickup_datetime", to_timestamp(col("pickup_datetime")))
    .withColumn("dropoff_datetime", to_timestamp(col("dropoff_datetime")))
    
    # Convert IDs to Integers
    .withColumn("pulocationid", col("pulocationid").cast(IntegerType()))
    .withColumn("dolocationid", col("dolocationid").cast(IntegerType()))
    
    # Remove duplicates based on unique trip identifiers
    .dropDuplicates(["hvfhs_license_num", "pickup_datetime", "pulocationid"])
    
    # Add a metadata column for the Silver processing time
    .withColumn("silver_processed_time", current_timestamp())
)

# 4. Write to Silver Table
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "abfss://silver@stlinggarprojectdev001.dfs.core.windows.net/nyc_ridehailing") # Correct existing table location
    .saveAsTable(silver_table_name)) # Registered name in Unity Catalog

print(f"Step 2 Complete: {spark.read.table(silver_table_name).count()} rows cleaned and moved to Silver.")